In [36]:
import os
import glob

import numpy as np
import polars as pl
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve, auc

In [2]:
WORKING_DIR = '/group/pmc021/amunif/epi-thesis/workflow/08_HepG2/'
OUTPUT_DIR = os.path.join(WORKING_DIR, 'output', 'logistic_regression', 'all_histone')

In [3]:
# Check CUDA device
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cuda device


# Load Dataset

## Load Dataset with label value_1

In [8]:
# Load dataset
gene_pl = pl.read_parquet(os.path.join(WORKING_DIR, 'dataset', 'gene_w_label_value_1.parquet'))
gene_pl.head(5)

gene_id,H3K4me3,H3K4me3_count,H3K9ac,H3K9ac_count,H3K9me3,H3K9me3_count,H3K27ac,H3K27ac_count,H3K27me3,H3K27me3_count,value_1,label
str,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64,f64,i32
"""XLOC_000001""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
"""XLOC_000003""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
"""XLOC_000006""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0888452,0
"""XLOC_000007""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,4.04743,1
"""XLOC_000008""","[0.0, 0.0, … 0.0]",6,"[0.0, 0.0, … 0.0]",3,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",2,"[0.0, 0.0, … 0.0]",0,26.7934,1


In [9]:
# Check the label distribution
gene_pl.group_by("label").len()

label,len
i32,u32
1,11077
0,11077


## Reformat the data based on the histone markers used

In [23]:
# Define the marker used
# markers = ['H3K4me3', 'H3K9me3', 'H3K27me3']
markers = ['H3K4me3', 'H3K9ac', 'H3K9me3', 'H3K27ac', 'H3K27me3']

In [24]:
processed_arrays = []

for col in markers:
    print(f"Processing {col}")
    arrays = gene_pl[col].to_list()
    stacked = np.vstack(arrays)
    processed_arrays.append(stacked)

# Concatenate all processed arrays horizontally
X = np.hstack(processed_arrays)

Processing H3K4me3
Processing H3K9ac
Processing H3K9me3
Processing H3K27ac
Processing H3K27me3


In [25]:
X.shape

(22154, 20000)

In [30]:
y = gene_pl['label'].to_numpy()

In [31]:
y.shape

(22154,)

## Split into train, validation, test

In [33]:
# Split the dataset into training, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.666, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

In [34]:
print("Train set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Test set shape:", X_test.shape, y_test.shape)

Train set shape: (7399, 20000) (7399,)
Validation set shape: (7377, 20000) (7377,)
Test set shape: (7378, 20000) (7378,)


In [38]:
# Create and train the model
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [39]:
# Get metrics for validation set
val_pred = model.predict(X_val)
val_metrics = classification_report(y_val, val_pred)

In [40]:
# Get metrics for test set
test_pred = model.predict(X_test)
test_metrics = classification_report(y_test, test_pred)

In [43]:
print(val_metrics)

              precision    recall  f1-score   support

           0       0.68      0.78      0.73      3689
           1       0.74      0.63      0.68      3688

    accuracy                           0.71      7377
   macro avg       0.71      0.71      0.71      7377
weighted avg       0.71      0.71      0.71      7377



In [44]:
print(test_metrics)

              precision    recall  f1-score   support

           0       0.69      0.78      0.73      3689
           1       0.75      0.64      0.69      3689

    accuracy                           0.71      7378
   macro avg       0.72      0.71      0.71      7378
weighted avg       0.72      0.71      0.71      7378

